In [1]:
import chromadb
import os
from dotenv import load_dotenv
from chromadb.utils import embedding_functions
from edgar import *
import yfinance as yf
from bs4 import BeautifulSoup
import requests
import re
import time

Connect to the Chroma Vector DB

In [2]:
# Path to .env location containing the Chroma API keys
env_path = os.path.join("../data", ".env")

# Parse the .env file and retrieve the API keys
if load_dotenv(env_path):
    print("✅ Environment variables loaded from data/.env")
else:
    print("❌ Failed to load data/.env - Check if the file exists!")

CF_CLIENT_ID = os.getenv("CF-ACCESS-CLIENT-ID")
CF_CLIENT_SECRET = os.getenv("CF-ACCESS-CLIENT-SECRET")
CHROMA_URL = os.getenv("CHROMA_URL")

print(f"ID: {CF_CLIENT_ID}")
print(f"Secret: {CF_CLIENT_SECRET}")
print(f"Chroma URL: {CHROMA_URL}")


✅ Environment variables loaded from data/.env
ID: 15703ac0e5797dac885c851f20afe6cb.access
Secret: 9db6c201a502a20d0aa9eae4dac06d36b2f6a29b46201c20df2f5174263ddf57
Chroma URL: chroma.taskcomply.com


In [3]:
# 2. Setup the Client with Custom Headers
client = chromadb.HttpClient(
    host=CHROMA_URL,                              # Your Cloudflare URL
    port=443,                                     # Standard HTTPS port
    ssl=True,                                     # MUST be True for Cloudflare
    headers={
        "CF-Access-Client-Id": CF_CLIENT_ID,
        "CF-Access-Client-Secret": CF_CLIENT_SECRET
    },
)

# 3. Test the connection
print(f"Connection Heartbeat: {client.heartbeat()}")

Connection Heartbeat: 1775470293022587790


Determine embedding function and create 2 streams:
- Fundamentals
- Sentiment

In [4]:
# 4. Use a lightweight embedding function
# Default is 'all-MiniLM-L6-v2' which is great for financial headlines
emb_fn = embedding_functions.DefaultEmbeddingFunction()

# 3. Create the two streams
fundamentals_col = client.get_or_create_collection(
    name="stream1_fundamentals",
    embedding_function=emb_fn
)

sentiment_col = client.get_or_create_collection(
    name="stream2_sentiment",
    embedding_function=emb_fn
)

In [5]:
collections = client.list_collections()

print(collections)

[Collection(name=stream1_fundamentals), Collection(name=stream2_sentiment)]


Remove old experimental data

In [6]:
# 1. Wipe the old collections
for col_name in ["stream1_fundamentals", "stream2_sentiment"]:
    try:
        client.delete_collection(name=col_name)
        print(f"🔥 Deleted collection: {col_name}")
    except Exception as e:
        print(f"ℹ️ {col_name} didn't exist or already deleted.")

# 2. Re-initialize them fresh
emb_fn = chromadb.utils.embedding_functions.DefaultEmbeddingFunction()

fundamentals_col = client.get_or_create_collection(name="stream1_fundamentals", embedding_function=emb_fn)
sentiment_col = client.get_or_create_collection(name="stream2_sentiment", embedding_function=emb_fn)

print("✅ Server collections are reset and ready for the clean run.")


🔥 Deleted collection: stream1_fundamentals
🔥 Deleted collection: stream2_sentiment
✅ Server collections are reset and ready for the clean run.


In [7]:

col = client.get_collection(name="stream2_sentiment")
print(col.count())


0


2. Stream 2: Ingesting Sentiment (The "Market Pulse")
For the sentiment stream, we want the most recent headlines. Since Tesla (TSLA) just had a delivery miss on April 2nd, and Apple (AAPL) just celebrated its 50th anniversary on April 1st, there is plenty of fresh data.

In [8]:

def get_verified_article_data(url, target_ticker, company_name):
    """
    Final Master's Level Scraper:
    1. Full-page scan for ticker/name density.
    2. 'Anchor Zone' check (Nvidia must be in the first 20% or the Title).
    3. 'Stop Marker' extraction to prevent appended articles.
    """
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
        response = requests.get(url, headers=headers, timeout=10)
        if response.status_code != 200:
            return None

        soup = BeautifulSoup(response.text, 'html.parser')

        # 1. CLEAN THE ENTIRE SOUP (Remove non-article noise)
        for noise in soup.find_all(["script", "style", "nav", "footer", "aside", "header"]):
            noise.decompose()

        # 2. FULL BODY TEXT SCAN
        entire_page_text = soup.get_text(separator=" ").upper()

        # 3. DENSITY & ANCHOR CHECK
        # How many times is our stock mentioned?
        ticker_count = entire_page_text.count(target_ticker.upper())
        name_count = entire_page_text.count(company_name.upper())
        total_mentions = ticker_count + name_count

        # Where is it mentioned? (Kill the 'Bitcoin' comparison articles)
        anchor_zone = entire_page_text[:int(len(entire_page_text) * 0.20)]
        page_title = soup.find('title').get_text().upper() if soup.find('title') else ""

        # RELEVANCE LOGIC:
        # Pass if in Title OR (Mentioned early AND at least 3 times total)
        is_relevant = False
        if target_ticker.upper() in page_title or company_name.upper() in page_title:
            is_relevant = True
        elif (target_ticker.upper() in anchor_zone or company_name.upper() in anchor_zone) and total_mentions >= 3:
            is_relevant = True

        if not is_relevant:
            return None

        # 4. EXTRACTION (Finding the 'Meat' of the article)
        best_container = None
        max_p = 0
        for div in soup.find_all(['div', 'article', 'main', 'section']):
            p_count = len(div.find_all('p', recursive=False))
            if p_count > max_p:
                max_p = p_count
                best_container = div

        if best_container:
            # 5. STOP MARKER LOGIC (Prevent 'Read Next' bleeding)
            stop_markers = ["read-next", "related-stories", "recirc-feed", "caas-footer", "article-separator"]
            clean_elements = []

            for el in best_container.find_all(['p', 'h1', 'h2', 'h3']):
                # Check class of element and its immediate parent
                el_classes = " ".join(el.get('class', [])) + " " + " ".join(el.parent.get('class', []))

                if any(marker in el_classes.lower() for marker in stop_markers):
                    break # Stop here: we've hit the appended article feed

                txt = el.get_text().strip()
                if len(txt) > 30: # Filter out social buttons/byline scraps
                    clean_elements.append(txt)

            full_text = "\n\n".join(clean_elements)

            # Final sanity check: Is it an actual article (>400 chars)?
            return full_text if len(full_text) > 400 else None

        return None
    except Exception:
        return None

In [9]:
# 1. THE SHARED ENGINE
def get_unified_news_pool(ticker_sym, company_name):
    """Exact same discovery logic for both Inspection and Writing."""
    # Standardize the queries here!
    search_queries = [
        ticker_sym,
        company_name,
        f"{company_name} stock"
    ]

    pool = {}
    for q in search_queries:
        # Requesting 25 to ensure we get a deep overlap
        results = yf.Search(q, news_count=25).news
        for item in results:
            pool[item['uuid']] = item

    print(f"🔎 Total unique articles discovered: {len(pool)}")
    print("-" * 50)

    return pool

In [10]:
def inspect_discovered_news(ticker_sym, company_name):

    # 1. Shared search
    discovered_pool = get_unified_news_pool(ticker_sym, company_name)

    # 2. Verification & Staging Phase
    verified_articles = []

    for uuid, item in discovered_pool.items():
        print(f"UUID: {uuid}")
        print(f"URL: {item['link']}")

        # Fetch and verify
        content = get_verified_article_data(item['link'], ticker_sym, company_name)

        if content:
            # Identify if it's a "Frankenstein" (likely appended articles)
            word_count = len(content.split())
            is_long = "⚠️ (Long/Possible Append)" if word_count > 1500 else ""

            verified_articles.append({
                "uuid": uuid,
                "headline": item['title'],
                "content": content,  # Keeping full content for your FinBERT later
                "body_snippet": content[:300].replace('\n', ' ') + "...",
                "word_count": word_count,
                "related": item.get('relatedTickers', []),
                "url": item['link']
            })
            print(f"✅ Verified: {item['title'][:50]}... [{word_count} words] {is_long}")
        else:
            print(f"❌ Rejected: {item['title'][:50]}...")

        time.sleep(0.4) # Respecting 2026 API thresholds

    # Return the full list of verified articles and the original pool
    return verified_articles, discovered_pool

# --- EXECUTION ---
target = "NVDA"
name = "Nvidia"
staged_data, original_pool = inspect_discovered_news(target, name)

# --- THE "DEEP INSPECTION" VIEW ---
print(f"\n--- INSPECTION REPORT: {len(staged_data)} VERIFIED ARTICLES ---")
for i, art in enumerate(staged_data):
    print(f"\n[{i+1}] {art['headline']} ({art['word_count']} words)")
    print(f"    URL: {art['url']}")
    print(f"    Related: {art['related']}")
    print(f"    Preview: {art['body_snippet']}")
    print("-" * 30)

🔎 Total unique articles discovered: 20
--------------------------------------------------
UUID: db18d05e-53b5-482b-9a38-da4a8d6d4888
URL: https://finance.yahoo.com/video/its-time-to-jump-in-with-palantir-and-these-2-stock-picks-100000513.html
❌ Rejected: It's 'time to jump in' with Palantir and these 2 s...
UUID: 05b391fe-15fc-3302-a3c1-e757f09855b9
URL: https://finance.yahoo.com/markets/stocks/articles/nvidia-nvda-narrative-shifting-us-060508868.html
✅ Verified: How The Nvidia (NVDA) Narrative Is Shifting With U... [488 words] 
UUID: 18999175-2a72-3fad-8d7f-72b84208724b
URL: https://finance.yahoo.com/m/18999175-2a72-3fad-8d7f-72b84208724b/the-great-rotation-won%27t-last.html
✅ Verified: The Great Rotation Won't Last Forever. 3 Growth St... [470 words] 
UUID: 2567b77a-08f4-32dd-97a3-b097d4b88bfd
URL: https://finance.yahoo.com/m/2567b77a-08f4-32dd-97a3-b097d4b88bfd/nasdaq-pullback%3A-3-stocks.html
✅ Verified: Nasdaq Pullback: 3 Stocks You'll Wish You Bought o... [533 words] 
UUID: 283b1

Check all articles are unique before writing to ChromaDB

In [11]:
all_headlines = [art['headline'] for art in staged_data]
unique_headlines = set(all_headlines)

print(f"Total Articles: {len(all_headlines)}")
print(f"Unique Headlines: {len(unique_headlines)}")

if len(all_headlines) != len(unique_headlines):
    print("⚠️ Warning: You have duplicate stories in your staging area!")

Total Articles: 19
Unique Headlines: 19


Write to Chroma DB

In [12]:
def get_bulk_verified_news_and_write_to_chromadb(ticker_sym, company_name, collection):

    discovered_news = get_unified_news_pool(ticker_sym, company_name)

    print(f"🔎 Unique articles found for {ticker_sym}: {len(discovered_news)}")

    for uuid, item in discovered_news.items():
        # 1. FAST CHECK: Skip if already in ChromaDB for q in search_queries:

        if collection.get(ids=[uuid])['ids']:
            # print(f"⏩ Skipping (Already in DB): {uuid}")
            continue

        # 2. UPDATED CALL: Passing company_name for the 'Deep Scan' logic
        content = get_verified_article_data(item['link'], ticker_sym, company_name)

        if content:
            # 3. METADATA ENRICHMENT: Save word count for future filtering
            word_count = len(content.split())

            collection.add(
                documents=[content],
                metadatas=[{
                    "ticker": ticker_sym,
                    "headline": item['title'],
                    "word_count": word_count,
                    "related": ", ".join(item.get('relatedTickers', [])),
                    "source": item.get('publisher', 'Unknown'),
                    "date_scraped": time.strftime("%Y-%m-%d") # Useful for tracking
                }],
                ids=[uuid]
            )
            print(f"✅ Verified & Saved: {item['title'][:55]}... ({word_count} words)")
        else:
            # No print here keeps your console clean for the actual saves
            pass

        time.sleep(0.5) # A slightly longer pause for the 'Full Scan' requests

In [13]:

# The "Discovery" List
stocks_to_track = [
    {"ticker": "NVDA", "name": "Nvidia"}
    #{"ticker": "TSLA", "name": "Tesla"},
    #{"ticker": "AAPL", "name": "Apple"}
]

for stock in stocks_to_track:
    print(f"\n--- Harvesting {stock['ticker']} ---")
    get_bulk_verified_news_and_write_to_chromadb(stock['ticker'], stock['name'], sentiment_col)



--- Harvesting NVDA ---
🔎 Total unique articles discovered: 20
--------------------------------------------------
🔎 Unique articles found for NVDA: 20
✅ Verified & Saved: How The Nvidia (NVDA) Narrative Is Shifting With US$1t ... (488 words)
✅ Verified & Saved: The Great Rotation Won't Last Forever. 3 Growth Stocks ... (470 words)
✅ Verified & Saved: Nasdaq Pullback: 3 Stocks You'll Wish You Bought on the... (533 words)
✅ Verified & Saved: NVIDIA CEO Jensen Huang Says AGI Is Here. If He’s Right... (500 words)
✅ Verified & Saved: Is Wheaton Precious Metals a Good Inflation Hedge?... (438 words)
✅ Verified & Saved: Brace for Impact: Why Oil Prices Could Be Extremely Vol... (511 words)
✅ Verified & Saved: The Federal Reserve's March and April Inflation Forecas... (459 words)
✅ Verified & Saved: Why IonQ Stock Plummeted 24.9% in March... (455 words)
✅ Verified & Saved: Be Wary of This S&P 500 Stock After Lamb Weston's Surpr... (271 words)
✅ Verified & Saved: Nvidia’s Trillion Dollar AI Go

In [15]:
col = client.get_collection(name="stream2_sentiment")
print(f"Total Articles successfully written to ChromaDB: {col.count()}")

Total Articles successfully written to ChromaDB: 19
